# 📘 LLM Zoomcamp — Homework 02: Vector Search

**Student:** Mohammed Sheikh-Noor  
**Course:** LLM Zoomcamp (DataTalksClub)  
**Repo:** https://github.com/mohanoor-ai/LLMZoomCamp  
**Date:** June 2026

In [2]:
import numpy as np
from embedder import Embedder

In [3]:
embedder = Embedder()

In [ ]:
dir(embedder)

In [5]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)
v[0]

np.float64(-0.02058203437252893)

1. Embedding a query (1 point)

-0.31

**-0.02**

0.12

0.44

---------------------------------------------------------------------------------------------------------------------------

In [6]:
import numpy as np

def cosine(v1, v2):
    return np.dot(v1, v2)

In [7]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [8]:
target = None

for doc in documents:
    if "07-sqlitesearch-vector.md" in doc["filename"]:
        target = doc
        break

target["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [9]:
doc_vec = embedder.encode(target["content"])
query_vec = embedder.encode("How does approximate nearest neighbor search work?")

cosine(query_vec, doc_vec)

np.float64(0.36107027225589694)

2. Cosine similarity (1 point)

0.07

**0.37**

0.68

0.92

---------------------------------------------------------------------------------------------------------------------------

In [11]:
from gitsource import chunk_documents

In [12]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [13]:
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)

In [14]:
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)

In [15]:
scores = X.dot(v)

In [16]:
best_idx = scores.argmax()
chunks[best_idx]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

3. Chunking and search by hand (1 point)


02-vector-search/lessons/03-embeddings-dataset.md

02-vector-search/lessons/06-rag-vector.md

**02-vector-search/lessons/07-sqlitesearch-vector.md**

02-vector-search/lessons/09-onnx-embedder.md

---------------------------------------------------------------------------------------------------------------------------

In [22]:
texts = [c["content"] for c in chunks]

payload = [
    {
        "filename": c["filename"],
        "start": c.get("start", 0)
    }
    for c in chunks
]

In [26]:
from minsearch import VectorSearch

index = VectorSearch()

In [27]:
index.fit(texts, payload)

In [29]:
import inspect
from minsearch import VectorSearch

print(inspect.signature(VectorSearch.search))

(self, query_vector, filter_dict=None, num_results=10, output_ids=False)


In [32]:
texts = [c["content"] for c in chunks]

In [33]:
query = "What metric do we use to evaluate a search engine?"
query_vec = embedder.encode(query)

In [34]:
X = embedder.encode_batch(texts)

In [36]:
query = "What metric do we use to evaluate a search engine?"
q = embedder.encode(query)

In [37]:
scores = X.dot(q)

In [38]:
best_idx = scores.argmax()
chunks[best_idx]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

4. Vector search with minsearch (1 point)

02-vector-search/lessons/04-vector-search.md

**04-evaluation/lessons/05-search-metrics.md**

04-evaluation/lessons/13-llm-as-judge.md

05-monitoring/lessons/04-metrics.md

---------------------------------------------------------------------------------------------------------------------------

In [39]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [40]:
query = "How do I store vectors in PostgreSQL?"

keyword_results = text_index.search(query, num_results=5)

In [41]:
query_vec = embedder.encode(query)

vector_scores = X.dot(query_vec)
top_idx = vector_scores.argsort()[-5:][::-1]

vector_results = [chunks[i] for i in top_idx]

In [42]:
keyword_files = {r["filename"] for r in keyword_results}
vector_files = {r["filename"] for r in vector_results}

vector_files - keyword_files

{'02-vector-search/lessons/08-pgvector.md'}

5. Text search vs vector search (1 point)

02-vector-search/lessons/01-intro.md

02-vector-search/lessons/02-embeddings.md

**02-vector-search/lessons/08-pgvector.md**

03-orchestration/lessons/05-rag.md

---------------------------------------------------------------------------------------------------------------------------

In [43]:
query = "How do I give the model access to tools?"

In [44]:
query_vec = embedder.encode(query)

vector_scores = X.dot(query_vec)
top_vec_idx = vector_scores.argsort()[-5:][::-1]

vector_results = [chunks[i] for i in top_vec_idx]

In [45]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [46]:
text_results = text_index.search(query, num_results=5)

In [47]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [48]:
results = rrf([vector_results, text_results])
results[0]["filename"]

'01-agentic-rag/lessons/13-function-calling.md'

6. Hybrid search (1 point)

01-agentic-rag/lessons/01-intro.md

**01-agentic-rag/lessons/13-function-calling.md**

01-agentic-rag/lessons/14-agentic-loop.md

01-agentic-rag/lessons/16-other-frameworks.md

---------------------------------------------------------------------------------------------------------------------------